In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd()

if (cwd / "requirements.txt").exists():
    ROOT = cwd
elif (cwd.parent / "requirements.txt").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Не найден корень проекта")

RAW = ROOT / "data" / "raw" / "binance"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

classified = pd.read_csv(
    PROCESSED / "btcusdt_1min_classified_2020_q1.csv"
)

for column in ["minute_start", "available_at"]:
    classified[column] = pd.to_datetime(
        classified[column], utc=True
    )

classified["tick_sign"] = pd.to_numeric(
    classified["tick_sign"], errors="raise"
)

MONTHS = ["2020-01", "2020-02", "2020-03"]

COLUMNS = [
    "trade_id",
    "price",
    "qty",
    "quote_qty",
    "time",
    "is_buyer_maker",
    "is_best_match",
]

print("Минутных строк:", len(classified))

Минутных строк: 131040


In [11]:
def aggregate_exchange_sides(path, month, chunksize=500_000):
    parts = []

    reader = pd.read_csv(
        path,
        header=None,
        names=COLUMNS,
        usecols=["qty", "time", "is_buyer_maker"],
        dtype={
            "qty": "float64",
            "time": "int64",
            "is_buyer_maker": "string",
        },
        chunksize=chunksize,
    )

    for number, chunk in enumerate(reader, start=1):
        flag = (
            chunk["is_buyer_maker"]
            .str.strip()
            .str.lower()
        )

        if not flag.isin(["true", "false"]).all():
            raise ValueError(f"{month}: некорректный maker-флаг")

        buyer_is_maker = flag.eq("true")

        chunk["minute_start"] = pd.to_datetime(
            chunk["time"], unit="ms", utc=True
        ).dt.floor("min")

        chunk["exchange_buy_btc"] = np.where(
            ~buyer_is_maker, chunk["qty"], 0.0
        )

        chunk["exchange_sell_btc"] = np.where(
            buyer_is_maker, chunk["qty"], 0.0
        )

        parts.append(
            chunk.groupby("minute_start", sort=False)[
                ["exchange_buy_btc", "exchange_sell_btc"]
            ].sum()
        )

        if number % 10 == 0:
            print(f"{month}: обработано порций — {number}")

    return (
        pd.concat(parts)
        .groupby(level="minute_start", sort=True)
        .sum()
        .reset_index()
    )


side_parts = []

for month in MONTHS:
    print(f"\nОбрабатываем {month}...")

    path = RAW / f"BTCUSDT-trades-{month}.csv"
    side_parts.append(aggregate_exchange_sides(path, month))

exchange_sides = pd.concat(side_parts, ignore_index=True)

assert exchange_sides["minute_start"].is_unique

print("\nБиржевые buy/sell-объёмы собраны.")


Обрабатываем 2020-01...
2020-01: обработано порций — 10
2020-01: обработано порций — 20
2020-01: обработано порций — 30

Обрабатываем 2020-02...
2020-02: обработано порций — 10
2020-02: обработано порций — 20
2020-02: обработано порций — 30

Обрабатываем 2020-03...
2020-03: обработано порций — 10
2020-03: обработано порций — 20
2020-03: обработано порций — 30
2020-03: обработано порций — 40
2020-03: обработано порций — 50

Биржевые buy/sell-объёмы собраны.


In [12]:
side_comparison = classified.merge(
    exchange_sides,
    on="minute_start",
    how="left",
    validate="one_to_one",
)

side_columns = ["exchange_buy_btc", "exchange_sell_btc"]

observed = side_comparison["trade_count"].gt(0)

assert side_comparison.loc[
    observed, side_columns
].notna().all().all()

side_comparison[side_columns] = (
    side_comparison[side_columns].fillna(0.0)
)

exchange_total = side_comparison[side_columns].sum(axis=1)

assert np.allclose(
    exchange_total,
    side_comparison["volume_btc"],
    rtol=1e-9,
    atol=1e-8,
)

valid = (
    observed
    & side_comparison["tick_sign"].isin([-1, 1])
)

side_comparison["matching_volume_btc"] = np.where(
    side_comparison["tick_sign"].eq(1),
    side_comparison["exchange_buy_btc"],
    np.where(
        side_comparison["tick_sign"].eq(-1),
        side_comparison["exchange_sell_btc"],
        np.nan,
    ),
)

side_comparison["opposite_volume_btc"] = np.where(
    valid,
    side_comparison["volume_btc"]
    - side_comparison["matching_volume_btc"],
    np.nan,
)

split = pd.Timestamp("2020-03-01", tz="UTC")
test_end = pd.Timestamp("2020-04-01", tz="UTC")

samples = {
    "train_jan_feb": valid
        & side_comparison["minute_start"].lt(split),
    "test_march": valid
        & side_comparison["minute_start"].ge(split)
        & side_comparison["minute_start"].lt(test_end),
}

rows = []

for name, mask in samples.items():
    frame = side_comparison.loc[mask]
    total = frame["volume_btc"].sum()

    rows.append({
        "sample": name,
        "classified_minutes": len(frame),
        "volume_btc": total,
        "matching_volume_pct": (
            100 * frame["matching_volume_btc"].sum() / total
        ),
        "opposite_volume_pct": (
            100 * frame["opposite_volume_btc"].sum() / total
        ),
    })

classification_summary = pd.DataFrame(rows)

print("Проверка сохранения объёма: OK\n")
print(classification_summary.round(6).to_string(index=False))

side_comparison.to_csv(
    PROCESSED / "minute_trade_side_comparison.csv",
    index=False,
)

classification_summary.to_csv(
    TABLES / "minute_tick_rule_classification.csv",
    index=False,
)

Проверка сохранения объёма: OK

       sample  classified_minutes   volume_btc  matching_volume_pct  opposite_volume_pct
train_jan_feb               85985 3.300998e+06             58.89335             41.10665
   test_march               44512 3.789769e+06             57.50501             42.49499


In [13]:
import sys
import json
import importlib

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.exchange_vpin as exchange_module

importlib.reload(exchange_module)

with (ROOT / "research_config.json").open(
    encoding="utf-8"
) as file:
    config = json.load(file)

bucket_volume = float(config["bucket_volume_btc"])
window = int(config["vpin_window"])

start_time = classified.loc[
    classified["tick_sign"].isin([-1, 1])
    & classified["volume_btc"].gt(0),
    "minute_start",
].min()

paths = [
    RAW / f"BTCUSDT-trades-{month}.csv"
    for month in MONTHS
]

exchange_vpin, accounting = (
    exchange_module.build_exchange_buckets(
        paths=paths,
        bucket_volume=bucket_volume,
        start_time=start_time,
        window=window,
    )
)

print("\nНачало отсчёта:", start_time)
print(pd.Series(accounting).to_string())

Читаем BTCUSDT-trades-2020-01.csv...
Читаем BTCUSDT-trades-2020-02.csv...
Читаем BTCUSDT-trades-2020-03.csv...

Начало отсчёта: 2020-01-01 00:01:00+00:00
source_volume_btc           7.090767e+06
source_buy_volume_btc       3.500151e+06
complete_buckets            6.444000e+03
residual_volume_btc         1.126827e+02
residual_buy_volume_btc     5.539455e+01
residual_sell_volume_btc    5.728815e+01


In [14]:
baseline = pd.read_csv(
    PROCESSED / "btcusdt_tr_vpin_2020_q1.csv"
)

baseline["available_at"] = pd.to_datetime(
    baseline["available_at"], utc=True
)

assert len(exchange_vpin) == len(baseline)

assert np.array_equal(
    exchange_vpin["bucket_id"].to_numpy(),
    baseline["bucket_id"].to_numpy(),
)

assert (
    exchange_vpin["available_at"].reset_index(drop=True)
    == baseline["available_at"].reset_index(drop=True)
).all()

tolerance = 1e-7

assert exchange_vpin[
    ["buy_volume_btc", "sell_volume_btc"]
].ge(-tolerance).all().all()

assert exchange_vpin["oi"].between(
    -tolerance, 1 + tolerance
).all()

assert exchange_vpin["exchange_vpin"].dropna().between(
    -tolerance, 1 + tolerance
).all()

assert np.isclose(
    exchange_vpin["buy_volume_btc"].sum()
    + accounting["residual_buy_volume_btc"],
    accounting["source_buy_volume_btc"],
    rtol=1e-10,
    atol=1e-7,
)

expected_total = classified.loc[
    classified["tick_sign"].isin([-1, 1]),
    "volume_btc",
].sum()

assert np.isclose(
    accounting["source_volume_btc"],
    expected_total,
    rtol=1e-9,
    atol=1e-7,
)

print("Сопоставимость с базовыми бакетами: OK")
print("Число полных бакетов:", len(exchange_vpin))
print(
    "Доступных значений Exchange-side VPIN:",
    int(exchange_vpin["exchange_vpin"].notna().sum()),
)

print(
    exchange_vpin["exchange_vpin"]
    .describe()
    .round(6)
    .to_string()
)

exchange_vpin.to_csv(
    PROCESSED / "btcusdt_exchange_vpin_2020_q1.csv",
    index=False,
)

Сопоставимость с базовыми бакетами: OK
Число полных бакетов: 6444
Доступных значений Exchange-side VPIN: 6395
count    6395.000000
mean        0.115846
std         0.023811
min         0.067535
25%         0.099618
50%         0.113766
75%         0.127747
max         0.260199


## Прогнозное сравнение Exchange-side VPIN

Используем те же обучающие и тестовые наблюдения,
что в базовом эксперименте.

Новую серию присоединяем по идентификатору последнего
доступного бакета, сохранённому в часовой выборке.

Параметры VPIN и горизонт прогноза не изменяются.
Март не используется для оценки коэффициентов моделей.

In [15]:
import statsmodels.api as sm

TARGET = "rv_next_1h_pct"

train_base = pd.read_csv(
    PROCESSED / "forecast_train.csv"
)

test_base = pd.read_csv(
    PROCESSED / "forecast_test.csv"
)

for frame in [train_base, test_base]:
    for column in [
        "timestamp",
        "target_end",
        "signal_available_at",
    ]:
        frame[column] = pd.to_datetime(frame[column], utc=True)

exchange_lookup = exchange_vpin[
    ["bucket_id", "available_at", "exchange_vpin"]
].rename(columns={
    "available_at": "exchange_signal_available_at"
})


def attach_exchange_signal(frame):
    result = frame.merge(
        exchange_lookup,
        on="bucket_id",
        how="left",
        validate="many_to_one",
        sort=False,
    )

    assert len(result) == len(frame)
    assert result["timestamp"].equals(
        frame["timestamp"].reset_index(drop=True)
    )

    assert result["exchange_vpin"].notna().all()

    assert (
        result["exchange_signal_available_at"]
        == result["signal_available_at"]
    ).all()

    assert (
        result["exchange_signal_available_at"]
        <= result["timestamp"]
    ).all()

    return result


train_ex = attach_exchange_signal(train_base)
test_ex = attach_exchange_signal(test_base)

print("Обучающих строк:", len(train_ex))
print("Тестовых строк:", len(test_ex))
print("Временная привязка Exchange-side VPIN: OK")

Обучающих строк: 1378
Тестовых строк: 740
Временная привязка Exchange-side VPIN: OK


In [16]:
FEATURE_SETS = {
    "TR_only": [
        "tr_vpin",
    ],
    "Exchange_only": [
        "exchange_vpin",
    ],
    "Controls": [
        "rv_prev_1h_pct",
        "log_volume_prev_1h",
    ],
    "Controls_TR": [
        "rv_prev_1h_pct",
        "log_volume_prev_1h",
        "tr_vpin",
    ],
    "Controls_U2": [
        "rv_prev_1h_pct",
        "log_volume_prev_1h",
        "u2_vpin",
    ],
    "Controls_EX": [
        "rv_prev_1h_pct",
        "log_volume_prev_1h",
        "exchange_vpin",
    ],
    "Controls_TR_EX": [
        "rv_prev_1h_pct",
        "log_volume_prev_1h",
        "tr_vpin",
        "exchange_vpin",
    ],
}

extended_predictions = test_ex[
    ["timestamp", "target_end", TARGET]
].copy()

extended_fits = {}
coefficient_rows = []

for name, features in FEATURE_SETS.items():
    X_train = sm.add_constant(
        train_ex[features].astype(float),
        has_constant="add",
    )

    X_test = sm.add_constant(
        test_ex[features].astype(float),
        has_constant="add",
    ).reindex(columns=X_train.columns)

    assert np.isfinite(X_train.to_numpy()).all()
    assert np.isfinite(X_test.to_numpy()).all()

    if np.linalg.matrix_rank(X_train.to_numpy()) < X_train.shape[1]:
        raise ValueError(f"{name}: линейно зависимые признаки")

    fit = sm.OLS(
        train_ex[TARGET].astype(float),
        X_train,
        missing="raise",
    ).fit()

    extended_fits[name] = fit
    extended_predictions[name] = fit.predict(X_test)

    for term, value in fit.params.items():
        coefficient_rows.append({
            "model": name,
            "term": term,
            "coefficient": float(value),
        })

extended_predictions["Persistence"] = (
    test_ex["rv_prev_1h_pct"].to_numpy()
)

extended_predictions["Train_mean"] = float(
    train_ex[TARGET].mean()
)

extended_coefficients = pd.DataFrame(coefficient_rows)

print("Модели обучены.")

Модели обучены.


In [17]:
old_predictions = pd.read_csv(
    PROCESSED / "hourly_test_predictions_baseline.csv"
)

old_predictions["timestamp"] = pd.to_datetime(
    old_predictions["timestamp"], utc=True
)

assert extended_predictions["timestamp"].equals(
    old_predictions["timestamp"]
)

for name in [
    "TR_only",
    "Controls",
    "Controls_TR",
    "Controls_U2",
    "Persistence",
    "Train_mean",
]:
    assert np.allclose(
        extended_predictions[name],
        old_predictions[name],
        rtol=1e-10,
        atol=1e-10,
    ), f"Изменились базовые прогнозы: {name}"

print("Базовые прогнозы воспроизведены без изменений: OK")

Базовые прогнозы воспроизведены без изменений: OK


In [18]:
model_names = [
    *FEATURE_SETS.keys(),
    "Persistence",
    "Train_mean",
]

actual = extended_predictions[TARGET].to_numpy(dtype=float)

rows = []

for name in model_names:
    forecast = extended_predictions[name].to_numpy(dtype=float)
    error = forecast - actual

    rows.append({
        "model": name,
        "n_test": len(actual),
        "MSE": float(np.mean(error ** 2)),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "MAE": float(np.mean(np.abs(error))),
        "bias": float(np.mean(error)),
        "negative_predictions": int((forecast < 0).sum()),
    })

extended_metrics = pd.DataFrame(rows).set_index("model")

for benchmark, suffix in [
    ("Controls", "controls"),
    ("Persistence", "persistence"),
]:
    benchmark_mse = extended_metrics.loc[benchmark, "MSE"]

    extended_metrics[f"MSE_gain_vs_{suffix}_pct"] = (
        100 * (1 - extended_metrics["MSE"] / benchmark_mse)
    )

print(
    extended_metrics
    .sort_values("RMSE")
    .round(6)
    .to_string()
)

print("\nКОЭФФИЦИЕНТЫ НОВЫХ МОДЕЛЕЙ:")
print(
    extended_coefficients.loc[
        extended_coefficients["model"].isin([
            "Exchange_only",
            "Controls_EX",
            "Controls_TR_EX",
        ])
    ].round(6).to_string(index=False)
)

extended_predictions.to_csv(
    PROCESSED / "hourly_test_predictions_extended.csv",
    index=False,
)

extended_metrics.to_csv(
    TABLES / "extended_forecast_metrics.csv"
)

extended_coefficients.to_csv(
    TABLES / "extended_regression_coefficients.csv",
    index=False,
)

                n_test       MSE      RMSE       MAE      bias  negative_predictions  MSE_gain_vs_controls_pct  MSE_gain_vs_persistence_pct
model                                                                                                                                      
Persistence        740  1.423754  1.193212  0.487237  0.000281                     0                 20.174700                     0.000000
Controls_U2        740  1.752057  1.323653  0.562291 -0.485128                     0                  1.767804                   -23.058976
Controls_TR_EX     740  1.763282  1.327886  0.559772 -0.476551                     0                  1.138437                   -23.847405
Controls_TR        740  1.770869  1.330740  0.570727 -0.497050                     0                  0.713065                   -24.380284
Controls           740  1.783587  1.335510  0.576553 -0.502451                     0                  0.000000                   -25.273567
Controls_EX        7